In [1]:
# =========================
# Imports
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
# =========================
# Load Data
# =========================
df = pd.read_csv("/Users/ameypatil/Desktop/Tata Power/data/2024 data/monsoon_2024.csv")  # change path

# Ensure sorted

# Datetime handling
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime')

# Feature
df['hour'] = df['datetime'].dt.hour

# Drop NaN
df = df.dropna()

df.head()

,datetime,wdLoad,temp,rh,year,month,hour
0,2024-06-01 00:00:00+05:30,412.224507,29.5,83.0,2024,6,0
1,2024-06-01 00:15:00+05:30,408.488763,29.4,84.0,2024,6,0
2,2024-06-01 00:30:00+05:30,401.194049,29.3,84.0,2024,6,0
3,2024-06-01 00:45:00+05:30,397.532262,29.2,85.0,2024,6,0
4,2024-06-01 01:00:00+05:30,388.329308,29.2,84.0,2024,6,1


In [3]:
# =========================
# Define Variables
# =========================

# Outcome
Y = df['wdLoad'].values

# Multi-treatment (temp + humidity)
T = df[['temp', 'rh']].values

# Confounders
X = df[['hour']].values  # add more if needed

In [4]:
# =========================
# Cross-Fitting DML
# =========================

kf = KFold(n_splits=10, shuffle=True, random_state=42)

Y_res = np.zeros_like(Y)
T_res = np.zeros_like(T)

for train_idx, test_idx in kf.split(X):
    
    X_train, X_test = X[train_idx], X[test_idx]
    Y_train, Y_test = Y[train_idx], Y[test_idx]
    T_train, T_test = T[train_idx], T[test_idx]
    
    # Models
    model_y = RandomForestRegressor(n_estimators=100, random_state=42)
    model_t = RandomForestRegressor(n_estimators=100, random_state=42)
    
    # Fit
    model_y.fit(X_train, Y_train)
    model_t.fit(X_train, T_train)
    
    # Residuals
    Y_res[test_idx] = Y_test - model_y.predict(X_test)
    T_res[test_idx] = T_test - model_t.predict(X_test)

# Final stage
final_model = LinearRegression()
final_model.fit(T_res, Y_res)

beta_temp = final_model.coef_[0]
beta_rh = final_model.coef_[1]

print("ATE (Temp):", beta_temp)
print("ATE (RH):", beta_rh)

ATE (Temp): 6.520724501696405
ATE (RH): -0.38197645592254004
